# 纯 NumPy CNN 猫狗分类器：完整流程

一个 Notebook 包含数据预处理、数据加载、CNN 基础层、模型、优化器、训练、验证和独立测试集评估。

当前结构：`Conv-ReLU-Pool → Conv-ReLU-Pool → Flatten → Affine → Output`。

## 0. 使用说明

1. 按顺序运行单元。
2. 已有预处理数据时无需再次执行批处理。
3. 小批次链路测试约需 1 秒。
4. 完整训练约需 20～30 分钟，需手动打开运行开关。
5. 猫标签为 0，狗标签为 1。

In [ ]:
# Notebook 运行环境初始化
from pathlib import Path
import os
import sys


def find_project_root():
    """从当前目录向上寻找项目根目录。"""
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "implementations").is_dir() and (candidate / "data").is_dir():
            return candidate
        nested = candidate / "猫狗分类器"
        if (nested / "implementations").is_dir() and (nested / "data").is_dir():
            return nested
    raise FileNotFoundError("找不到猫狗分类器项目根目录")


PROJECT_ROOT = find_project_root()
CODE_DIR = PROJECT_ROOT / "implementations" / "numpy_baseline"

# 让 NumPy 原始版中的本地模块导入保持有效
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

# 模拟原 Python 文件路径，使 Path(__file__) 逻辑保持有效
__file__ = str(CODE_DIR / "complete_notebook.py")
os.chdir(CODE_DIR)

print("项目根目录：", PROJECT_ROOT)
print("当前工作目录：", Path.cwd())


## 1. 图片批量预处理

In [ ]:
from pathlib import Path
from PIL import Image, ImageOps,UnidentifiedImageError

IMAGE_SIZE = (128,128)

MAX_IMAGES = None

# 当前要处理的数据集
DATASET_SPLIT = "test"

# 当前项目根目录：猫狗分类器
project_root = Path(__file__).resolve().parents[2]

# 原始测试集：
# cat-dog-classifier/data/raw/test
source_root = (
    project_root
    / "data"
    / "raw"
    / DATASET_SPLIT
)

# 预处理测试集：
# cat-dog-classifier/data/processed/test
target_root = (
    project_root
    / "data"
    / "processed"
    / DATASET_SPLIT
)

class_names = ["cats","dogs"]
valid_suffixes = {".jpg",".jpeg",".png"}

def preprocess_image(source_path,target_path):

    with Image.open(source_path) as image:
        image = image.convert("RGB")

    processed_image = ImageOps.fit(
        image,
        IMAGE_SIZE,
        method=Image.Resampling.LANCZOS,
        centering=(0.5,0.5),
    )

    processed_image.save(
        target_path,
        format = "JPEG",
        quality = 95
    )

def run_batch_preprocessing():
    for class_name in class_names:
        source_class_dir = source_root/class_name
        target_class_dir = target_root/class_name

        target_class_dir.mkdir(parents=True,exist_ok=True)

        image_paths = sorted(
            path
            for path in source_class_dir.iterdir()
            if path.suffix.lower() in valid_suffixes
        )

        if MAX_IMAGES is not None:
            image_paths = image_paths[:MAX_IMAGES]

        success_count = 0
        failed_count = 0

        for index,source_path in enumerate(image_paths,start=1):
            target_path = target_class_dir/f"{source_path.stem}.jpg"

            try:
                preprocess_image(source_path,target_path)
                success_count +=1
            except(UnidentifiedImageError,OSError) as error:
                failed_count +=1
                print(f"处理失败：{source_path}，原因:{error}")

            if index % 500 ==0:
                print(f"{class_name} 已处理 {index}/{len(image_paths)}")
        print(
            f"{class_name} 完成："
            f"成功 {success_count} 张，失败 {failed_count} 张"
        )

    print("预处理数据保存位置：", target_root)


In [ ]:
# 已有预处理数据时保持 False
RUN_PREPROCESSING = False

if RUN_PREPROCESSING:
    run_batch_preprocessing()
else:
    print("跳过批量预处理")

## 2. 数据加载与分层划分

In [ ]:
from pathlib import Path

import numpy as np
from PIL import Image

VALID_SUFFIXES = {".jpg",".jpeg",".png"}

def collect_dataset(dataset_root):
    class_to_label = {
        "cats":0,
        "dogs":1,
    }
    image_path = []
    labels = []

    for class_name,label in class_to_label.items():
        class_dir = dataset_root / class_name

        if not class_dir.is_dir():
            raise FileNotFoundError(
                f"找不到类别目录：{class_dir}"
            )

        class_paths = sorted(
            path
            for path in class_dir.iterdir()
            if path.suffix.lower() in VALID_SUFFIXES
        )

        image_path.extend(class_paths)
        labels.extend([label] * len(class_paths))

    return (
        np.array(image_path,dtype=object),
        np.array(labels,dtype=np.int64)
    )

def load_image(image_path):
    with Image.open(image_path) as image:
        image = image.convert("RGB")

        image_array = (
            np.asarray(
                image,
                dtype=np.float32,
            )
            /255.0
        )

    image_array = image_array.transpose(2,0,1)

    if image_array.shape != (3,128,128):
        raise ValueError(
            f"{image_path} 的形状错误："
            f"{image_array.shape}"
        )       
    return image_array

def load_batch(batch_paths,batch_labels):
    images = [
        load_image(path)
        for path in batch_paths
    ]

    batch_images = np.stack(
        images,
        axis=0,
    )

    batch_labels=np.asarray(
        batch_labels,
        dtype=np.int64,
    )

    return batch_images,batch_labels

def batch_iterator(
    image_paths,
    labels,
    batch_size,
    random_generator,
    shuffle=True,
):
    """
    逐批读取数据，避免把全部图片放进内存。
    """

    indices = np.arange(len(image_paths))

    if shuffle:
        random_generator.shuffle(indices)

    for start in range(
        0,
        len(indices),
        batch_size,
    ):
        end = start + batch_size
        batch_indices = indices[start:end]

        yield load_batch(
            image_paths[batch_indices],
            labels[batch_indices],
        )

def stratified_split(
    image_paths,
    labels,
    validation_ratio,
    random_generator,
):
    """
    分别划分猫和狗，确保训练集、验证集类别平衡。
    """

    train_indices = []
    validation_indices = []

    for label in np.unique(labels):
        class_indices = np.flatnonzero(
            labels == label
        )

        random_generator.shuffle(class_indices)

        validation_count = int(
            len(class_indices)
            * validation_ratio
        )

        validation_indices.extend(
            class_indices[:validation_count]
        )

        train_indices.extend(
            class_indices[validation_count:]
        )

    train_indices = np.array(
        train_indices,
        dtype=np.int64,
    )

    validation_indices = np.array(
        validation_indices,
        dtype=np.int64,
    )

    # 避免训练数据仍然是先猫后狗
    random_generator.shuffle(train_indices)
    random_generator.shuffle(validation_indices)

    return (
        image_paths[train_indices],
        labels[train_indices],
        image_paths[validation_indices],
        labels[validation_indices],
    )

## 3. 纯 NumPy CNN 基础层

In [ ]:
import numpy as np

def im2col(input_data,filter_height,filter_width,stride = 1,pad = 0,):
    N,C,H,W = input_data.shape
    output_height =(H + 2*pad - filter_height)//stride + 1 
    output_width =(W + 2*pad - filter_width)//stride + 1 

    padded_image = np.pad(
        input_data,
        (
            (0,0),
            (0,0),
            (pad,pad),
            (pad,pad),
        ),
        mode="constant",
    )

    col = np.zeros(
        (
            N,C,filter_height,filter_width,output_height,output_width
        ),
        dtype=input_data.dtype
    )
    for y in range(filter_height):
        y_max = y + stride*output_height

        for x in range(filter_width):
            x_max = x + stride*output_width

            col[:,:,y,x,:,:] = padded_image[:,:,y:y_max:stride,x:x_max:stride,]

    col = col.transpose(0,4,5,1,2,3)
    col = col.reshape(N * output_height * output_width,-1)

    return col

def col2im(
    col,
    input_shape,
    filter_height,
    filter_width,
    stride=1,
    pad=0,
):
    """
    把 im2col 格式的数据恢复成图片形状。

    注意：卷积窗口重叠的区域会进行梯度累加。
    """

    N, C, H, W = input_shape

    output_height = (
        H + 2 * pad - filter_height
    ) // stride + 1

    output_width = (
        W + 2 * pad - filter_width
    ) // stride + 1

    # 恢复 im2col 展开前的多维排列
    col = col.reshape(
        N,
        output_height,
        output_width,
        C,
        filter_height,
        filter_width,
    )

    col = col.transpose(0, 3, 4, 5, 1, 2)

    # stride - 1 是为了容纳最后一个滑动窗口
    image = np.zeros(
        (
            N,
            C,
            H + 2 * pad + stride - 1,
            W + 2 * pad + stride - 1,
        ),
        dtype=col.dtype,
    )

    for y in range(filter_height):
        y_max = y + stride * output_height

        for x in range(filter_width):
            x_max = x + stride * output_width

            # 使用 +=，因为不同窗口可能覆盖同一个像素
            image[
                :,
                :,
                y:y_max:stride,
                x:x_max:stride,
            ] += col[:, :, y, x, :, :]

    # 去掉之前添加的 padding
    return image[
        :,
        :,
        pad:H + pad,
        pad:W + pad,
    ]


class Convolution:
    def __init__(self,weight,bias,stride = 1,pad = 0):
        self.weight = weight
        self.bias = bias
        self.stride = stride
        self.pad = pad

        self.x = None 
        self.col = None
        self.col_weight = None 

        self.dweight = None
        self.dbias = None

    def forward(self,x):
        filter_number,C,FH,FW = self.weight.shape
        N,C,H,W, = x.shape

        output_height = (H + 2*self.pad -FH)//self.stride + 1
        output_width = (W + 2*self.pad -FW)//self.stride + 1

        col = im2col(x,FH,FW,self.stride,self.pad)

        col_weight = self.weight.reshape(filter_number,-1).T

        out = col @ col_weight + self.bias

        out =out.reshape(N,output_height,output_width,filter_number)
        out = out.transpose(0,3,1,2)

        self.x = x
        self.col = col
        self.col_weight = col_weight

        return out

    def backward(self,dout):
        filter_number, C, FH, FW = self.weight.shape

        # 将输出梯度转换成矩阵乘法需要的形状
        dout = dout.transpose(0, 2, 3, 1)
        dout = dout.reshape(-1, filter_number)

        # 偏置梯度：对所有样本和空间位置求和
        self.dbias = np.sum(dout, axis=0)

        # 权重梯度
        dweight = self.col.T @ dout

        # 恢复成卷积核原来的形状
        self.dweight = dweight.T.reshape(
            filter_number,
            C,
            FH,
            FW,
        )

        # 传给输入的梯度
        dcol = dout @ self.col_weight.T

        dx = col2im(
            dcol,
            self.x.shape,
            FH,
            FW,
            self.stride,
            self.pad,
        )

        return dx

class MaxPooling:
    def __init__(self,pool_height =2,pool_width=2,stride = 2,pad=0):
        self.pool_height = pool_height
        self.pool_width = pool_width
        self.stride = stride
        self.pad = pad

        self.x =None
        self.argmax = None

    def forward(self,x):
        N,C,H,W = x.shape

        output_height = (H + 2*self.pad - self.pool_height)//self.stride + 1
        output_width = (W + 2*self.pad - self.pool_width)//self.stride + 1

        col = im2col(x,self.pool_height,self.pool_width,self.stride,self.pad)

        pool_size =(self.pool_height*self.pool_width)

        col = col.reshape(-1,pool_size)

        argmax = np.argmax(col,axis=1)

        out = np.max(col,axis=1)

        out = out.reshape(N,output_height,output_width,C)

        out = out.transpose(0,3,1,2)

        self.x = x
        self.argmax = argmax

        return out
    
    def backward(self,dout):
        dout = dout.transpose(0,2,3,1)

        pool_size =(self.pool_height*self.pool_width)

        dmax = np.zeros(
            (
                self.argmax.size,
                pool_size,
            ),
            dtype=dout.dtype,
        )
        dmax[
            np.arange(self.argmax.size),
            self.argmax,
        ] = dout.reshape(-1)

        N,C,H,W =self.x.shape

        output_height = (
            H + 2 * self.pad - self.pool_height
        ) // self.stride + 1

        output_width = (
            W + 2 * self.pad - self.pool_width
        ) // self.stride + 1

        dcol = dmax.reshape(N*output_height*output_width,C*pool_size)

        dx = col2im(dcol,self.x.shape,self.pool_height,self.pool_width,self.stride,self.pad)

        return dx

class ReLU:

    def __init__(self):
        self.mask = None

    def forward(self,x):
        self.mask = x <=0

        out = x.copy()
        out[self.mask] = 0

        return out

    def backward(self,dout):

        dx = dout.copy()
        dx[self.mask] = 0

        return dx

class Flatten:

    def __init__(self):
        self.original_shape = None 

    def forward(self,x):
        self.original_shape = x.shape
        batch_size = x.shape[0]

        return x.reshape(batch_size,-1)

    def backward(self,dout):
        return dout.reshape(self.original_shape)

class Affine:

    def __init__(self,weight,bias):
        self.weight = weight
        self.bias = bias

        self.x=None

        self.dweight = None 
        self.dbias = None

    def forward(self,x):
        self.x = x
        out = x @ self.weight + self.bias

        return out
    
    def backward(self,dout):
        dx = dout @ self.weight.T
        self.dweight = self.x.T @ dout
        self.dbias = np.sum(dout,axis=0)

        return dx

def softmax(x):
    if x.ndim ==1:
        shifted_x = x-np.max(x)
        exp_x = np.exp(shifted_x)

        return exp_x / np.sum(exp_x)

    shifted_x = x -np.max(x,axis=1,keepdims=True)
    exp_x=np.exp(shifted_x)
    probabilities = exp_x /np.sum(exp_x,axis=1,keepdims=True)

    return probabilities

def cross_entropy_loss(probabilities,labels):
    if probabilities.ndim ==1:
        probabilities = probabilities.reshape(1,-1)

    labels = np.asarray(labels,dtype=np.int64).reshape(-1)

    batch_size = probabilities.shape[0]

    if len(labels) !=batch_size:
        raise ValueError("标签数量必须与样本数量相同")

    correct_probabilities = probabilities[np.arange(batch_size),labels]

    loss = -np.mean(np.log(correct_probabilities +1e-7))

    return loss

class SoftmaxAndLoss:
    def __init__(self):
        self.loss = None 
        self.probabilities =None
        self.lables = None

    def forward(self,scores,lables):
        self.lables = np.asarray(lables,dtype=np.int64).reshape(-1)

        self.probabilities = softmax(scores)
        self.loss = cross_entropy_loss(self.probabilities,self.lables)

        return self.loss

    def backward(self,dout =1.0):
        batch_size = self.lables.shape[0]

        dx =self.probabilities.copy()

        dx[np.arange(batch_size),self.lables] -=1

        dx *= dout /batch_size

        return dx

    


## 4. SGD 优化器

In [ ]:
class SGD:
    def __init__(self,learning_rate = 0.005):
        self.learning_rate = learning_rate

    def update(self,params,gradients):
        """
        新参数 = 旧参数 - 学习率 * 梯度
        """
        for name in params:
            if name not in gradients:
                raise KeyError(
                    f"找不到参数 {name} 的梯度"
                )

            if (
                params[name].shape!=gradients[name].shape
            ):
                raise ValueError(f"{name} 的参数与梯度形状不一致")

            params[name] -=(
                self.learning_rate
                *gradients[name]
            )

        

## 5. CNN 模型

In [ ]:
from collections import OrderedDict
from pathlib import Path

import numpy as np

class CatVsDog:
    def __init__(self,seed = 42):
        rng = np.random.default_rng(seed)

        input_channels = 3
        input_height = 128
        input_width = 128

        filter1_number = 8
        filter2_number =16
        filter_size = 3

        final_height = input_height//4
        final_width = input_width//4

        affine_input_size = filter2_number*final_height*final_width

        self.params={}
        #第一个卷积层权重 
        self.params["W1"] = (
            rng.standard_normal(
                (
                    filter1_number,
                    input_channels,
                    filter_size,
                    filter_size,
                )
            )
            *np.sqrt(
                2.0
                /(
                    input_channels
                    *filter_size
                    *filter_size
                )
            )
        ).astype(np.float32)

        self.params["b1"] = np.zeros(
            filter1_number,
            dtype=np.float32,
        )
        #第二个卷积层权重
        self.params["W2"] = (
            rng.standard_normal(
                (
                    filter2_number,
                    filter1_number,
                    filter_size,
                    filter_size,
                )
            )
            * np.sqrt(
                2.0
                / (
                    filter1_number
                    * filter_size
                    * filter_size
                )
            )
        ).astype(np.float32)

        self.params["b2"] = np.zeros(
            filter2_number,
            dtype=np.float32,
        )
        #全链接层权重
        self.params["W3"] = (
            rng.standard_normal(
                (affine_input_size, 2)
            )
            * np.sqrt(1.0 / affine_input_size)
        ).astype(np.float32)

        self.params["b3"] = np.zeros(
            2,
            dtype=np.float32,
        )

        self.layers = OrderedDict()

        self.layers["Conv1"] = Convolution(
            self.params["W1"],
            self.params["b1"],
            stride=1,
            pad=1,
        )

        self.layers["ReLU1"] = ReLU()

        self.layers["Pool1"] = MaxPooling(
            pool_height=2,
            pool_width=2,
            stride=2,
        )

        self.layers["Conv2"] = Convolution(
            self.params["W2"],
            self.params["b2"],
            stride=1,
            pad=1,
        )

        self.layers["ReLU2"] = ReLU()

        self.layers["Pool2"] = MaxPooling(
            pool_height=2,
            pool_width=2,
            stride=2,
        )

        self.layers["Flatten"] = Flatten()

        self.layers["Affine"] = Affine(
            self.params["W3"],
            self.params["b3"],
        )

        self.loss_layer = SoftmaxAndLoss()

    def predict(self,x):
        for layer in self.layers.values():
            x = layer.forward(x)

        return x

    def loss(self,x,lables):
        scores = self.predict(x)

        return self.loss_layer.forward(scores,lables)

    def accuracy(self,x,lables):
        scores = self.predict(x)
        predictions = np.argmax(scores,axis=1)

        return np.mean(predictions == lables)

    def gradient(self,x,lables):
        self.loss(x,lables)
        dout = self.loss_layer.backward()

        for layer in reversed(list(self.layers.values())):
            dout = layer.backward(dout)

        gradients = {
            "W1": self.layers["Conv1"].dweight,
            "b1": self.layers["Conv1"].dbias,
            "W2": self.layers["Conv2"].dweight,
            "b2": self.layers["Conv2"].dbias,
            "W3": self.layers["Affine"].dweight,
            "b3": self.layers["Affine"].dbias,
        }

        return gradients
    
    def load_parameters(self, model_path):
    

        model_path = Path(model_path)

        if not model_path.is_file():
            raise FileNotFoundError(
                f"找不到模型文件：{model_path}"
            )

        with np.load(model_path) as saved_params:
            for name in self.params:
                if name not in saved_params:
                    raise KeyError(
                        f"模型文件缺少参数：{name}"
                    )

                loaded_parameter = saved_params[name]

                if (
                    loaded_parameter.shape
                    != self.params[name].shape
                ):
                    raise ValueError(
                        f"{name} 形状不一致："
                        f"模型需要 {self.params[name].shape}，"
                        f"文件中为 {loaded_parameter.shape}"
                    )

            # [...] 表示原地复制到现有数组
                self.params[name][...] = (
                    loaded_parameter
                )
    

## 6. 真实图片小批次训练测试

损失应总体下降，用于确认数据、前向传播、反向传播和 SGD 已连通。

In [ ]:
def run_training_chain_test():
    from pathlib import Path

    import numpy as np



    project_root = Path(__file__).resolve().parents[2]

    dataset_root = (
        project_root
        / "data"
        / "processed"
        / "train"
    )

    image_paths, labels = collect_dataset(
        dataset_root
    )

    print("图片总数：", len(image_paths))
    print("猫的数量：", np.sum(labels == 0))
    print("狗的数量：", np.sum(labels == 1))


    # 选择两张猫图和两张狗图
    cat_indices = np.flatnonzero(labels == 0)[:2]
    dog_indices = np.flatnonzero(labels == 1)[:2]

    batch_indices = np.concatenate(
        [cat_indices, dog_indices]
    )

    # 打乱这四张图片的顺序
    rng = np.random.default_rng(seed=42)
    rng.shuffle(batch_indices)

    batch_images, batch_labels = load_batch(
        image_paths[batch_indices],
        labels[batch_indices],
    )

    print("Batch 图片形状：", batch_images.shape)
    print("Batch 标签：", batch_labels)


    model = CatVsDog(seed=42)

    optimizer = SGD(
        learning_rate=0.005
    )

    initial_loss = model.loss(
        batch_images,
        batch_labels,
    )

    print("初始损失：", initial_loss)


    # 反复学习同一个 mini-batch
    for step in range(1, 11):
        gradients = model.gradient(
            batch_images,
            batch_labels,
        )

        optimizer.update(
            model.params,
            gradients,
        )

        current_loss = model.loss(
            batch_images,
            batch_labels,
        )

        print(
            f"第 {step:02d} 次更新，"
            f"损失：{current_loss:.6f}"
        )


    final_loss = model.loss(
        batch_images,
        batch_labels,
    )

    print("最终损失：", final_loss)

    assert batch_images.shape == (4, 3, 128, 128)
    assert batch_labels.shape == (4,)
    assert final_loss < initial_loss

    print("真实图片 SGD 更新测试通过！")

In [ ]:
run_training_chain_test()

## 7. 完整训练与验证

训练数据自动按类别分层划分为 80% 训练集和 20% 验证集。

In [ ]:
import time
from pathlib import Path

import numpy as np



# -------------------------
# 训练配置
# -------------------------

SEED = 42
VALIDATION_RATIO = 0.2 #验证数据的比例 
BATCH_SIZE = 8
LEARNING_RATE = 0.005
EPOCHS = 10

# 第一次调试只使用少量数据
# 确认程序正常后改成 None
MAX_TRAIN_SAMPLES = None
MAX_VALIDATION_SAMPLES = None


def limit_dataset(
    image_paths,
    labels,
    max_samples,
):
    """调试时限制数据数量。"""

    if max_samples is None:
        return image_paths, labels

    max_samples = min(
        max_samples,
        len(image_paths),
    )

    return (
        image_paths[:max_samples],
        labels[:max_samples],
    )


def evaluate(
    model,
    image_paths,
    labels,
    batch_size,
):
    """
    在验证集上计算平均损失和准确率。

    验证过程只做前向传播，不更新参数。
    """

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    # 验证时不需要随机打乱
    validation_rng = np.random.default_rng(
        seed=0
    )

    iterator = batch_iterator(
        image_paths,
        labels,
        batch_size=batch_size,
        random_generator=validation_rng,
        shuffle=False,
    )

    for batch_images, batch_labels in iterator:
        loss = model.loss(
            batch_images,
            batch_labels,
        )

        predictions = np.argmax(
            model.loss_layer.probabilities,
            axis=1,
        )

        current_batch_size = len(batch_labels)

        total_loss += (
            float(loss)
            * current_batch_size
        )

        total_correct += np.sum(
            predictions == batch_labels
        )

        total_samples += current_batch_size

    average_loss = total_loss / total_samples
    accuracy = total_correct / total_samples

    return average_loss, accuracy


def save_parameters(model, save_path):
    """把所有模型参数保存为一个 npz 文件。"""

    save_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    np.savez(
        save_path,
        **model.params,
    )


def train_main():
    project_root = (
        Path(__file__).resolve().parents[2]
    )

    dataset_root = (
        project_root
        / "data"
        / "processed"
        / "train"
    )

    output_dir = project_root / "outputs" / "numpy_baseline"
    best_model_path = (
        output_dir
        / "best_model_numpy.npz"
    )

    rng = np.random.default_rng(
        seed=SEED
    )

    # 收集全部数据
    image_paths, labels = collect_dataset(
        dataset_root
    )

    # 分层划分训练集和验证集
    (
        train_paths,
        train_labels,
        validation_paths,
        validation_labels,
    ) = stratified_split(
        image_paths,
        labels,
        validation_ratio=VALIDATION_RATIO,
        random_generator=rng,
    )

    # 调试阶段限制样本数
    train_paths, train_labels = limit_dataset(
        train_paths,
        train_labels,
        MAX_TRAIN_SAMPLES,
    )

    (
        validation_paths,
        validation_labels,
    ) = limit_dataset(
        validation_paths,
        validation_labels,
        MAX_VALIDATION_SAMPLES,
    )

    print(
        "训练集：",
        len(train_paths),
        "验证集：",
        len(validation_paths),
    )

    print(
        "训练集猫/狗：",
        np.sum(train_labels == 0),
        "/",
        np.sum(train_labels == 1),
    )

    print(
        "验证集猫/狗：",
        np.sum(validation_labels == 0),
        "/",
        np.sum(validation_labels == 1),
    )

    model = CatVsDog(seed=SEED)

    optimizer = SGD(
        learning_rate=LEARNING_RATE
    )

    best_validation_accuracy = -1.0
    training_start = time.time()

    for epoch in range(1, EPOCHS + 1):
        epoch_start = time.time()

        total_loss = 0.0
        total_correct = 0
        total_samples = 0

        train_iterator = batch_iterator(
            train_paths,
            train_labels,
            batch_size=BATCH_SIZE,
            random_generator=rng,
            shuffle=True,
        )

        total_batches = (
            len(train_paths)
            + BATCH_SIZE
            - 1
        ) // BATCH_SIZE

        for batch_number, (
            batch_images,
            batch_labels,
        ) in enumerate(
            train_iterator,
            start=1,
        ):
            # gradient 内部会先完成一次前向传播
            gradients = model.gradient(
                batch_images,
                batch_labels,
            )

            # 读取本次前向传播的损失和概率
            batch_loss = float(
                model.loss_layer.loss
            )

            predictions = np.argmax(
                model.loss_layer.probabilities,
                axis=1,
            )

            # 使用梯度更新全部参数
            optimizer.update(
                model.params,
                gradients,
            )

            current_batch_size = len(
                batch_labels
            )

            total_loss += (
                batch_loss
                * current_batch_size
            )

            total_correct += np.sum(
                predictions == batch_labels
            )

            total_samples += current_batch_size

            if (
                batch_number % 20 == 0
                or batch_number == total_batches
            ):
                print(
                    f"\rEpoch {epoch}/{EPOCHS} "
                    f"Batch {batch_number}/"
                    f"{total_batches}",
                    end="",
                    flush=True,
                )

        train_loss = (
            total_loss / total_samples
        )

        train_accuracy = (
            total_correct / total_samples
        )

        validation_loss, validation_accuracy = (
            evaluate(
                model,
                validation_paths,
                validation_labels,
                BATCH_SIZE,
            )
        )

        epoch_seconds = (
            time.time() - epoch_start
        )

        print()
        print(
            f"训练损失：{train_loss:.4f}，"
            f"训练准确率："
            f"{train_accuracy * 100:.2f}%"
        )

        print(
            f"验证损失："
            f"{validation_loss:.4f}，"
            f"验证准确率："
            f"{validation_accuracy * 100:.2f}%"
        )

        print(
            f"本轮耗时："
            f"{epoch_seconds:.2f} 秒"
        )

        # 只保存验证准确率最高的模型
        if (
            validation_accuracy
            > best_validation_accuracy
        ):
            best_validation_accuracy = (
                validation_accuracy
            )

            save_parameters(
                model,
                best_model_path,
            )

            print(
                "已保存最佳模型：",
                best_model_path,
            )

    total_seconds = (
        time.time() - training_start
    )

    print()
    print(
        "训练完成，最佳验证准确率："
        f"{best_validation_accuracy * 100:.2f}%"
    )

    print(
        f"总耗时：{total_seconds:.2f} 秒"
    )




In [ ]:
# 完整训练约需 20～30 分钟；确认配置后改成 True
RUN_FULL_TRAINING = False

if RUN_FULL_TRAINING:
    train_main()
else:
    print("未启动完整训练；请确认 EPOCHS、BATCH_SIZE 和样本限制")

## 8. 独立测试集评估

加载验证准确率最高的 `best_model_numpy.npz`，报告测试损失、准确率和混淆矩阵。

In [ ]:
from pathlib import Path

import numpy as np



TEST_BATCH_SIZE = 16


def evaluation_main():
    project_root = (
        Path(__file__).resolve().parents[2]
    )

    test_root = (
        project_root
        / "data"
        / "processed"
        / "test"
    )

    model_path = (
        project_root
        / "outputs"
        / "numpy_baseline"
        / "best_model_numpy.npz"
    )

    # 收集测试图片和标签
    test_paths, test_labels = collect_dataset(
        test_root
    )

    print("测试集总数：", len(test_paths))
    print(
        "猫的数量：",
        np.sum(test_labels == 0),
    )
    print(
        "狗的数量：",
        np.sum(test_labels == 1),
    )

    # 创建相同结构的网络
    model = CatVsDog(seed=42)

    # 使用训练保存的最佳参数覆盖随机参数
    model.load_parameters(model_path)

    print("已加载模型：", model_path)

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    # 行表示真实类别，列表示预测类别
    #
    #           预测猫  预测狗
    # 真实猫
    # 真实狗
    confusion_matrix = np.zeros(
        (2, 2),
        dtype=np.int64,
    )

    random_generator = np.random.default_rng(
        seed=0
    )

    test_iterator = batch_iterator(
        test_paths,
        test_labels,
        batch_size=TEST_BATCH_SIZE,
        random_generator=random_generator,
        shuffle=False,
    )

    total_batches = (
        len(test_paths)
        + TEST_BATCH_SIZE
        - 1
    ) // TEST_BATCH_SIZE

    for batch_number, (
        batch_images,
        batch_labels,
    ) in enumerate(
        test_iterator,
        start=1,
    ):
        # 只做前向传播，不更新参数
        batch_loss = model.loss(
            batch_images,
            batch_labels,
        )

        predictions = np.argmax(
            model.loss_layer.probabilities,
            axis=1,
        )

        current_batch_size = len(
            batch_labels
        )

        total_loss += (
            float(batch_loss)
            * current_batch_size
        )

        total_correct += np.sum(
            predictions == batch_labels
        )

        total_samples += current_batch_size

        # 把每个样本加入混淆矩阵
        np.add.at(
            confusion_matrix,
            (batch_labels, predictions),
            1,
        )

        if (
            batch_number % 50 == 0
            or batch_number == total_batches
        ):
            print(
                f"\r测试进度："
                f"{batch_number}/{total_batches}",
                end="",
                flush=True,
            )

    average_loss = total_loss / total_samples
    accuracy = total_correct / total_samples

    cat_accuracy = (
        confusion_matrix[0, 0]
        / np.sum(confusion_matrix[0])
    )

    dog_accuracy = (
        confusion_matrix[1, 1]
        / np.sum(confusion_matrix[1])
    )

    print()
    print(
        f"测试损失：{average_loss:.4f}"
    )
    print(
        f"测试准确率：{accuracy * 100:.2f}%"
    )

    print()
    print("混淆矩阵：")
    print("          预测猫  预测狗")
    print(
        f"真实猫："
        f"{confusion_matrix[0, 0]:6d} "
        f"{confusion_matrix[0, 1]:6d}"
    )
    print(
        f"真实狗："
        f"{confusion_matrix[1, 0]:6d} "
        f"{confusion_matrix[1, 1]:6d}"
    )

    print()
    print(
        f"猫识别正确率："
        f"{cat_accuracy * 100:.2f}%"
    )
    print(
        f"狗识别正确率："
        f"{dog_accuracy * 100:.2f}%"
    )

    assert total_samples == 5000
    assert np.sum(confusion_matrix) == 5000




In [ ]:
# 已有 best_model_numpy.npz 时可改成 True
RUN_FINAL_EVALUATION = False

if RUN_FINAL_EVALUATION:
    evaluation_main()
else:
    print("未启动独立测试集评估")

## 当前基线结果

- 独立测试准确率：**76.68%**
- 猫识别正确率：**72.20%**
- 狗识别正确率：**81.16%**